# Volatility Dynamics 03b — calibrating the Module 03 precursor scan

Claude review (**VD 010**). Module 03 scanned every numeric field × 5 pre-event lags, kept each field's best lag and ranked by a discovery score with **no reference distribution**.

03b reruns the identical scan (reproduced exactly; verified against Module 03's own code on synthetic data) on **day-relabel placebo events**: each real q99 onset keeps its asset and exact UTC slot, but its UTC day is permuted. Same-day onsets move together, so event count, clock mix, cross-asset synchronicity and within-day clustering are preserved.

Reported per field: observed score, placebo mean / q95, per-field $p$ and **family-wise** $p$ (vs. the maximum score over all fields in each placebo run). Also for the pre-registered fixed summary (mean percentile over the five lags; no best-lag pick).

**Interpretation boundary.** Placebos fall on ordinary moments, real onsets on volatile ones, so volatility-type fields *should* beat this null. 03b answers "is the ranking beyond selection noise?", not "is anything beyond volatility?" (that is 04b).

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_q99_precursors as qp
from data_engineering.modeling import volatility_vd_b_core as vb
from data_engineering.modeling import volatility_q99_precursor_null as pn
START = os.getenv('VD3B_START', '').strip() or None
END = os.getenv('VD3B_END', '').strip() or None
ASSETS = [x.strip().upper() for x in os.getenv('VD3B_ASSETS', '').split(',') if x.strip()]
N_PERM = int(os.getenv('VD3B_N_PERM', str(pn.DEFAULT_N_PERM)))
SEED = int(os.getenv('VD3B_SEED', str(pn.DEFAULT_SEED)))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '03b'
OUT.mkdir(parents=True, exist_ok=True)
print('core', vb.VERSION, vb.selftest())
print('03b', pn.VERSION, pn.selftest())


In [ ]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

def load_frame(columns, start=None, end=None, assets=None):
    cols = list(dict.fromkeys(['state_time', 'underlying', 'return_5m'] + list(columns)))
    where = []
    if start:
        ts = pd.Timestamp(start)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
    if end:
        ts = pd.Timestamp(end)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time < TIMESTAMPTZ '{(ts + pd.Timedelta(days=1)).isoformat()}'")
    if assets:
        where.append('underlying IN (' + ','.join(repr(x) for x in assets) + ')')
    sql = ('SELECT ' + ', '.join(qident(c) for c in cols) + ' FROM time_series_feature_state'
           + ('' if not where else ' WHERE ' + ' AND '.join(where)) + ' ORDER BY underlying, state_time')
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        df = con.execute(sql).df()
    finally:
        con.close()
    if df.empty:
        raise RuntimeError('No rows found.')
    df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
    if df.duplicated(['underlying', 'state_time']).any():
        raise RuntimeError('Duplicate asset/timestamp rows.')
    print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())} | {df.state_time.min()} -> {df.state_time.max()}')
    return df.reset_index(drop=True)

def schema_columns():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.execute('DESCRIBE time_series_feature_state').df()
    finally:
        con.close()

schema = schema_columns()
predictors, inventory = qp.eligible_predictor_columns(schema)
print('eligible predictors (Module 03 rule):', len(predictors))
df = load_frame(predictors, START, END, ASSETS)


In [ ]:
grid = vb.build_grid(df)
thr = vb.q99_thresholds(grid)                    # full-sample, as Module 03
tail = vb.tail_flags(grid, thr)
event_rows = np.flatnonzero(vb.onset_flags(grid, tail))
eligible = vb.eligible_origins(grid, tail)
X = grid.feature_matrix(predictors)
print('asset-event onsets:', len(event_rows), '| unique timestamps:', len(np.unique(grid.time_ns[event_rows])))
print('valid days for relabelling:', len(vb.valid_days(grid)))


In [ ]:
observed = pn.scan(grid, X, event_rows, eligible)
observed.insert(0, 'feature', [predictors[i] for i in observed.feature_index])
ref = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '03' / '03_precursor_rankings.csv'
if ref.exists():
    r3 = pd.read_csv(ref)[['feature', 'discovery_score']].merge(observed[['feature', 'discovery_score']], on='feature', suffixes=('_module03', '_03b'))
    diff = (r3.discovery_score_module03 - r3.discovery_score_03b).abs()
    print(f'reproduction vs Module 03 output: {len(r3)} features, max |diff| = {diff.max():.2e} (non-zero only if the database changed since Module 03 ran)')
display(observed.sort_values('discovery_score', ascending=False).head(30).round(4))


In [ ]:
t0 = time.time()
S, W, counts = pn.placebo_null(grid, X, event_rows, eligible, n_perm=N_PERM, seed=SEED, progress_every=50)
print(f'{N_PERM} placebo scans in {time.time()-t0:.0f}s')
display(counts.describe().round(1))
calibrated = pn.calibrate(observed, predictors, S, W)
fw = pn.familywise_thresholds(S, W)
display(fw.round(4))
display(calibrated.head(40).round(4))
print('fields with family-wise p < 0.05 (discovery score):', int((calibrated.p_familywise_score < 0.05).sum()))


In [ ]:
inventory.to_csv(OUT / '03b_feature_inventory.csv', index=False)
pd.DataFrame([{'underlying': k, 'q99_threshold': v} for k, v in thr.items()]).to_csv(OUT / '03b_q99_thresholds.csv', index=False)
observed.to_csv(OUT / '03b_observed_rankings.csv', index=False)
calibrated.to_csv(OUT / '03b_calibrated_rankings.csv', index=False)
fw.to_csv(OUT / '03b_familywise_null.csv', index=False)
counts.to_csv(OUT / '03b_placebo_counts.csv', index=False)
meta = {
    'version': pn.VERSION, 'core_version': vb.VERSION, 'module': '03b_q99_precursor_null',
    'scan': 'Module 03 clock-only controls, rank lags -60,-30,-15,-10,-5 min, best-lag discovery score (reproduced exactly)',
    'null': 'day-relabel placebo: UTC day permuted, asset and exact slot kept; same-day events move together',
    'n_perm': N_PERM, 'seed': SEED, 'n_predictors': len(predictors),
    'n_asset_event_onsets': int(len(event_rows)), 'n_unique_event_times': int(len(np.unique(grid.time_ns[event_rows]))),
    'start': START or 'earliest_available', 'end': END or 'latest_available',
    'first_time': str(df.state_time.min()), 'last_time': str(df.state_time.max()), 'n_rows': int(len(df)),
}
(OUT / '03b_run_metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote', OUT)
